# 🏥 Sprint 1 — Foundation & Input Pipeline
**Project:** Autonomous Medical Research & Patient Triage Assistant
**Day:** 1 of 5 | **Sprint Goal:** Input pipeline with mode detection

## What we build
```
Message → Validate → Detect Mode → Parse Symptoms → [Emergency | Acknowledge]
```

## User Stories
- US-01: Patient sends symptoms via Telegram/WhatsApp → receives triage
- US-02: System detects patient vs clinician mode automatically

## Safety Rule
Every response must include the safety disclaimer.

In [ ]:
%pip install -q openai langchain langchain-openai langchain-core langgraph pinecone cohere python-dotenv tiktoken requests

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

# Uncomment if not using .env file:
# os.environ['OPENAI_API_KEY']   = 'sk-...'
# os.environ['PINECONE_API_KEY'] = 'pcsk_...'
# os.environ['COHERE_API_KEY']   = '...'

OPENAI_API_KEY   = os.getenv('OPENAI_API_KEY', '')
PINECONE_API_KEY = os.getenv('PINECONE_API_KEY', '')
COHERE_API_KEY   = os.getenv('COHERE_API_KEY', '')

assert OPENAI_API_KEY,   'OPENAI_API_KEY not set'
assert PINECONE_API_KEY, 'PINECONE_API_KEY not set'
assert COHERE_API_KEY,   'COHERE_API_KEY not set'

print('OpenAI  :', OPENAI_API_KEY[:12] + '...')
print('Pinecone:', PINECONE_API_KEY[:12] + '...')
print('Cohere  :', COHERE_API_KEY[:8] + '...')
print('All keys loaded successfully')

In [ ]:
from typing import TypedDict, Optional, List, Dict

class MedicalAgentState(TypedDict):
    # Sprint 1: Input
    session_id:        str
    ticket_id:         str
    initiated_at:      str
    channel:           str   # telegram or whatsapp
    chat_id:           str
    user_mode:         str   # patient or clinician
    raw_message:       str
    symptoms:          List[str]
    duration:          Optional[str]
    age:               Optional[str]
    medications:       List[str]
    status:            str
    errors:            List[str]
    # Sprint 2: Research
    raw_research:      Optional[str]
    research_chunks:   Optional[List[str]]
    pinecone_ids:      Optional[List[str]]
    # Sprint 3: Triage
    retrieved_chunks:  Optional[List[str]]
    reranked_chunks:   Optional[List[str]]
    urgency_level:     Optional[str]
    urgency_score:     Optional[int]
    differential:      Optional[List[str]]
    red_flags:         Optional[List[str]]
    retry_count:       int
    # Sprint 4: Report
    triage_report:     Optional[Dict[str, str]]
    nutrition_advice:  Optional[str]
    home_remedies:     Optional[str]
    follow_up_sent:    bool
    report_ready:      bool
    # Audit
    workflow_path:     List[str]

print('MedicalAgentState defined:', len(MedicalAgentState.__annotations__), 'fields')

In [ ]:
import re

CLINICIAN_KEYWORDS = [
    'i am a doctor', 'i am a nurse', 'i am a clinician', 'i am a pharmacist',
    'i am a physician', 'i am a midwife', 'my patient', 'the patient',
    'presenting with', 'chief complaint', 'differential', 'treatment protocol',
    'drug of choice', 'contraindication', 'community health worker', 'nhis provider'
]

def detect_user_mode(message: str) -> str:
    msg = message.lower()
    for kw in CLINICIAN_KEYWORDS:
        if kw in msg:
            return 'clinician'
    return 'patient'

def extract_symptoms(message: str) -> dict:
    dur_match = re.search(r'(\d+\s*(?:day|days|week|weeks|hour|hours))', message, re.IGNORECASE)
    age_match = re.search(r'(\d+)\s*(?:year|yr|years old)', message, re.IGNORECASE)
    duration  = dur_match.group(1) if dur_match else None
    age       = age_match.group(1) + ' years' if age_match else None
    symptoms  = [s.strip() for s in re.split(r'[,;]|\band\b', message) if len(s.strip()) > 3]
    return {'symptoms': symptoms[:10], 'duration': duration, 'age': age, 'medications': []}

# Quick test
print('Mode detection test:')
print(detect_user_mode('I have headache and fever'))           # -> patient
print(detect_user_mode('I am a doctor, my patient has malaria'))  # -> clinician
print('Symptom extraction test:')
info = extract_symptoms('I have headache, fever and weakness for 3 days')
print('Symptoms:', info['symptoms'][:3])
print('Duration:', info['duration'])

In [ ]:
RED_FLAG_SYMPTOMS = {
    'cardiac':      ['chest pain', 'cannot breathe', 'shortness of breath', 'palpitation'],
    'neurological': ['seizure', 'convulsion', 'fits', 'unconscious', 'stroke', 'face drooping'],
    'obstetric':    ['pregnant bleeding', 'baby not moving', 'labour pain severe'],
    'haemorrhage':  ['vomiting blood', 'blood in stool', 'bleeding heavily'],
    'sepsis':       ['temperature 40', 'not responding', 'shaking badly'],
}

def detect_red_flags(message: str) -> List[str]:
    msg      = message.lower()
    detected = []
    for category, keywords in RED_FLAG_SYMPTOMS.items():
        for kw in keywords:
            if kw in msg:
                detected.append(f'{category}: {kw}')
                break
    return detected

# Test
print('Red flag detection test:')
for msg in ['I have chest pain and cannot breathe', 'My child is having convulsion', 'I have headache']:
    flags = detect_red_flags(msg)
    print(f'{"EMERGENCY" if flags else "OK":12} | {msg[:50]}')
    for f in flags:
        print(f'  Red flag: {f}')

In [ ]:
import uuid
from datetime import datetime, timezone
from langgraph.graph import StateGraph, END

def validate_input_node(state: MedicalAgentState) -> MedicalAgentState:
    raw    = state.get('raw_message', '').strip()
    errors = list(state.get('errors', []))
    print(f'[VALIDATE] Input: {raw[:60]}')
    if not raw or len(raw) < 5:
        return {
            **state, 'status': 'error_invalid_input',
            'errors': errors + ['Message too short. Please describe your symptoms in more detail.'],
            'workflow_path': state.get('workflow_path', []) + ['validate_input']
        }
    now        = datetime.now(timezone.utc)
    ticket_id  = f'MED-{now.strftime("%Y%m%d")}-{str(uuid.uuid4())[:8].upper()}'
    session_id = str(uuid.uuid4())
    print(f'[VALIDATE] Ticket: {ticket_id}')
    return {
        **state, 'raw_message': raw[:5000], 'ticket_id': ticket_id,
        'session_id': session_id, 'initiated_at': now.isoformat(),
        'status': 'validated', 'errors': errors,
        'workflow_path': state.get('workflow_path', []) + ['validate_input']
    }

def parse_input_node(state: MedicalAgentState) -> MedicalAgentState:
    message = state['raw_message']
    mode    = detect_user_mode(message)
    info    = extract_symptoms(message)
    flags   = detect_red_flags(message)
    print(f'[PARSE] Mode: {mode} | Symptoms: {info["symptoms"][:2]} | Red flags: {len(flags)}')
    if flags:
        print(f'[PARSE] RED FLAGS: {flags}')
    return {
        **state, 'user_mode': mode, 'symptoms': info['symptoms'],
        'duration': info['duration'], 'age': info['age'],
        'medications': info['medications'], 'red_flags': flags,
        'status': 'parsed',
        'workflow_path': state.get('workflow_path', []) + ['parse_input']
    }

def emergency_node(state: MedicalAgentState) -> MedicalAgentState:
    flags = state.get('red_flags', [])
    print(f'[EMERGENCY] Fast-tracking emergency response | Flags: {flags}')
    emergency_msg = (
        'EMERGENCY - IMMEDIATE ACTION REQUIRED\n\n'
        'Based on your symptoms, you need emergency medical care immediately.\n\n'
        'What to do RIGHT NOW:\n'
        '- Call 112 (Nigeria Emergency) or go to the nearest A&E\n'
        '- Do NOT drive yourself - call someone or take a keke/taxi\n'
        '- Do NOT take any medication without supervision\n\n'
        'Nigeria Emergency Numbers:\n'
        '- Emergency: 112\n'
        '- LASAMBUS (Lagos): 08000-432584\n'
        '- FRSC: 122\n\n'
        'DISCLAIMER: This is not a substitute for professional medical care.'
    )
    return {
        **state, 'urgency_level': 'emergency', 'urgency_score': 10,
        'triage_report': {'emergency_response': emergency_msg},
        'report_ready': True, 'status': 'emergency_complete',
        'workflow_path': state.get('workflow_path', []) + ['emergency']
    }

def acknowledge_node(state: MedicalAgentState) -> MedicalAgentState:
    mode   = state['user_mode']
    ticket = state['ticket_id']
    if mode == 'clinician':
        ack = (
            f'Clinical Query Received | Reference: {ticket}\n'
            f'Mode: Clinician\n'
            f'Searching WHO guidelines, NHIS protocols, and PubMed evidence...\n'
            f'Differential diagnosis ready in ~60 seconds.'
        )
    else:
        ack = (
            f'Symptom Report Received | Reference: {ticket}\n'
            f'Reviewing your symptoms against Nigerian health guidelines...\n'
            f'Your triage assessment will be ready in ~60 seconds.\n'
            f'DISCLAIMER: This is not a substitute for professional medical advice.'
        )
    print(f'[ACK] {mode.upper()} | Ticket: {ticket}')
    print(ack)
    return {
        **state, 'status': 'acknowledged',
        'workflow_path': state.get('workflow_path', []) + ['acknowledge']
    }

def error_handler_node(state: MedicalAgentState) -> MedicalAgentState:
    print('PIPELINE ERROR:')
    for e in state.get('errors', []):
        print(f'  ERROR: {e}')
    return {
        **state, 'status': 'failed',
        'workflow_path': state.get('workflow_path', []) + ['error_handler']
    }

print('All Sprint 1 nodes defined')

In [ ]:
def route_after_validation(state):
    return 'error_handler' if 'error' in state.get('status', '') else 'parse_input'

def route_after_parsing(state):
    if state.get('red_flags'):
        print('[ROUTER] Red flags detected -> Emergency fast-track')
        return 'emergency'
    return 'acknowledge'

def build_sprint1_graph():
    g = StateGraph(MedicalAgentState)
    g.add_node('validate_input', validate_input_node)
    g.add_node('parse_input',    parse_input_node)
    g.add_node('emergency',      emergency_node)
    g.add_node('acknowledge',    acknowledge_node)
    g.add_node('error_handler',  error_handler_node)
    g.set_entry_point('validate_input')
    g.add_conditional_edges('validate_input', route_after_validation,
        {'parse_input': 'parse_input', 'error_handler': 'error_handler'})
    g.add_conditional_edges('parse_input', route_after_parsing,
        {'emergency': 'emergency', 'acknowledge': 'acknowledge'})
    g.add_edge('emergency',     END)
    g.add_edge('acknowledge',   END)
    g.add_edge('error_handler', END)
    return g.compile()

sprint1_graph = build_sprint1_graph()
print('Sprint 1 graph compiled')
print('Flow: validate_input -> parse_input -> [emergency | acknowledge] -> END')

In [ ]:
def create_medical_state(message: str, channel: str = 'telegram', chat_id: str = '000') -> MedicalAgentState:
    return MedicalAgentState(
        session_id='', ticket_id='', initiated_at='',
        channel=channel, chat_id=chat_id, user_mode='patient',
        raw_message=message, symptoms=[], duration=None, age=None,
        medications=[], status='pending', errors=[],
        raw_research=None, research_chunks=None, pinecone_ids=None,
        retrieved_chunks=None, reranked_chunks=None,
        urgency_level=None, urgency_score=None,
        differential=None, red_flags=None, retry_count=0,
        triage_report=None, nutrition_advice=None,
        home_remedies=None, follow_up_sent=False,
        report_ready=False, workflow_path=[]
    )

print('create_medical_state helper ready')

In [ ]:
# TEST 1: Normal patient
print('=' * 55)
print('TEST 1 - Patient: headache and fever for 3 days')
print('=' * 55)
r1 = sprint1_graph.invoke(create_medical_state('I have been having headache and fever for 3 days, I feel very weak'))
print(f'Status   : {r1["status"]}')
print(f'Mode     : {r1["user_mode"]}')
print(f'Ticket   : {r1["ticket_id"]}')
print(f'Symptoms : {r1["symptoms"][:3]}')
print(f'Path     : {" -> ".join(r1["workflow_path"])}')
assert r1['status'] == 'acknowledged', f'Expected acknowledged, got {r1["status"]}'
assert r1['user_mode'] == 'patient'
assert r1['ticket_id'].startswith('MED-')
print('TEST 1 PASSED\n')

# TEST 2: Clinician
print('TEST 2 - Clinician mode detection')
r2 = sprint1_graph.invoke(create_medical_state('I am a doctor. My patient is presenting with severe malaria.'))
print(f'Mode: {r2["user_mode"]}')
assert r2['user_mode'] == 'clinician'
print('TEST 2 PASSED\n')

# TEST 3: Emergency
print('TEST 3 - Emergency: chest pain')
r3 = sprint1_graph.invoke(create_medical_state('I have chest pain and cannot breathe, sweating a lot'))
print(f'Urgency  : {r3["urgency_level"]}')
print(f'Red flags: {r3["red_flags"]}')
assert r3['urgency_level'] == 'emergency'
assert r3['report_ready'] == True
print('TEST 3 PASSED\n')

# TEST 4: Invalid input
print('TEST 4 - Invalid: too short')
r4 = sprint1_graph.invoke(create_medical_state('hi'))
assert r4['status'] == 'failed'
print('TEST 4 PASSED\n')

print('ALL TESTS PASSED')
print('Sprint 1 DoD: MET - Ready for Sprint 2')

In [ ]:
import json, pathlib
output = dict(r1)
pathlib.Path('sprint1_medical_output.json').write_text(json.dumps(output, indent=2, default=str))
print('Sprint 1 output saved to sprint1_medical_output.json')
print('Load in Sprint 2 with: json.load(open("sprint1_medical_output.json"))')